# LNN Telemetry Engine

Implementation of a Liquid Neural Network (LNN) for detecting anomalies in CubeSat telemetry data. Runnable in Colab.


In [ ]:
# Install the Neural Circuit Policies (NCPs) library which provides the LNN/CfC implementations.
!pip install ncps -q


In [ ]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from typing import List, Optional, Sequence, Union, Tuple, Dict
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

from ncps.torch import CfC
from ncps.wirings import AutoNCP

# Ensure reproducability
torch.manual_seed(42)
np.random.seed(42)

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


## 1. Utilities
Metrics and matplotlib configurations.


In [ ]:
"""
utils/metrics.py — Hand-rolled classification metrics for the LNN Telemetry Engine.

Every function in this module is implemented from scratch using only NumPy.
The goal is to demonstrate a deep understanding of the math behind common
binary-classification metrics rather than simply calling sklearn.


* All inputs are coerced to 1-D NumPy int arrays so callers can pass plain
  Python lists or tensors without friction.
* Division-by-zero is handled gracefully — the affected metric returns 0.0
  instead of raising or returning NaN.
* Type hints and docstrings follow Google-style conventions.

Metrics implemented
-------------------
precision, recall, f1_score, accuracy, confusion_matrix,
optimize_threshold, classification_report

"""

from __future__ import annotations

from typing import List, Optional, Sequence, Union

import numpy as np

# Type alias — anything that can be treated as a label vector.
ArrayLike = Union[np.ndarray, List[int], Sequence[int]]


# Helper: coerce arbitrary inputs to flat int arrays

def _to_array(x: ArrayLike) -> np.ndarray:
    """Convert *x* to a 1-D NumPy integer array.

    This lets every public function accept plain Python lists, nested
    sequences, or existing ndarrays without duplicating conversion logic.
    """
    return np.asarray(x, dtype=int).ravel()


# Core confusion-matrix counts

def _counts(y_true: np.ndarray, y_pred: np.ndarray) -> tuple[int, int, int, int]:
    """Return (TP, FP, FN, TN) for binary classification.

    
    * True Positive  (TP): predicted 1, actual 1
    * False Positive (FP): predicted 1, actual 0
    * False Negative (FN): predicted 0, actual 1
    * True Negative  (TN): predicted 0, actual 0

    We compute these with simple boolean masks rather than looping — this is
    both faster and more readable than an explicit for-loop.
    """
    tp = int(np.sum((y_pred == 1) & (y_true == 1)))
    fp = int(np.sum((y_pred == 1) & (y_true == 0)))
    fn = int(np.sum((y_pred == 0) & (y_true == 1)))
    tn = int(np.sum((y_pred == 0) & (y_true == 0)))
    return tp, fp, fn, tn


# Precision

def precision(y_true: ArrayLike, y_pred: ArrayLike) -> float:
    """Compute precision = TP / (TP + FP).

    Precision answers: "Of everything we *predicted* as positive, how many
    actually were?"  A low precision means lots of false alarms.

    """
    y_true, y_pred = _to_array(y_true), _to_array(y_pred)
    tp, fp, _fn, _tn = _counts(y_true, y_pred)

    # Guard against division by zero — if the model never predicted positive,
    # precision is undefined; we return 0.0 by convention.
    denominator = tp + fp
    if denominator == 0:
        return 0.0

    return tp / denominator


# Recall

def recall(y_true: ArrayLike, y_pred: ArrayLike) -> float:
    """Compute recall = TP / (TP + FN).

    Recall answers: "Of all *actual* positives, how many did we catch?"
    A low recall means we're missing real anomalies.

    """
    y_true, y_pred = _to_array(y_true), _to_array(y_pred)
    tp, _fp, fn, _tn = _counts(y_true, y_pred)

    denominator = tp + fn
    if denominator == 0:
        return 0.0

    return tp / denominator


# F1 Score

def f1_score(y_true: ArrayLike, y_pred: ArrayLike) -> float:
    """Compute the F1 score = 2 · (P · R) / (P + R).

    The F1 score is the *harmonic mean* of precision and recall.  Unlike
    the arithmetic mean, the harmonic mean penalizes extreme imbalance —
    you can't get a high F1 by gaming only one of the two.

    """
    p = precision(y_true, y_pred)
    r = recall(y_true, y_pred)

    # If both precision and recall are zero, F1 is 0 (avoid 0/0).
    if p + r == 0:
        return 0.0

    # Harmonic mean formula: 2·P·R / (P + R)
    return 2.0 * p * r / (p + r)


# Accuracy

def accuracy(y_true: ArrayLike, y_pred: ArrayLike) -> float:
    """Compute accuracy = (TP + TN) / N.

    Simply counts how many predictions matched the ground truth.

    """
    y_true, y_pred = _to_array(y_true), _to_array(y_pred)

    n = len(y_true)
    if n == 0:
        return 0.0

    # Element-wise comparison → count matches → divide by total.
    return float(np.sum(y_true == y_pred)) / n


# Confusion Matrix

def confusion_matrix(y_true: ArrayLike, y_pred: ArrayLike) -> np.ndarray:
    """Build a 2×2 confusion matrix for binary classification.

    Layout (matches sklearn convention)::

        [[TN, FP],
         [FN, TP]]

    Row index = actual class, column index = predicted class.

    """
    y_true, y_pred = _to_array(y_true), _to_array(y_pred)
    tp, fp, fn, tn = _counts(y_true, y_pred)

    # Arrange in the standard [[TN, FP], [FN, TP]] layout.
    return np.array([[tn, fp],
                     [fn, tp]], dtype=int)


# Threshold Optimization

def optimize_threshold(
    y_true: ArrayLike,
    y_scores: Union[np.ndarray, List[float]],
    metric: str = "f1",
    thresholds: Optional[Union[np.ndarray, List[float]]] = None,
) -> tuple[float, float]:
    """Sweep thresholds to find the one that maximizes a chosen metric.

    Many models output continuous scores (e.g., sigmoid probabilities).  To
    convert these to binary predictions we pick a threshold *t* — scores ≥ t
    become 1, the rest become 0.  This function tries many thresholds and
    returns the best one.

    """
    # ── Map metric name → callable ────────────────────────────────────────
    metric_fn_map = {
        "f1": f1_score,
        "precision": precision,
        "recall": recall,
        "accuracy": accuracy,
    }

    if metric not in metric_fn_map:
        raise ValueError(
            f"Unknown metric '{metric}'. "
            f"Choose from {list(metric_fn_map.keys())}."
        )

    metric_fn = metric_fn_map[metric]

    # ── Coerce inputs ─────────────────────────────────────────────────────
    y_true = _to_array(y_true)
    y_scores = np.asarray(y_scores, dtype=float).ravel()

    # ── Default threshold grid: 0.10, 0.15, …, 0.90 ──────────────────────
    if thresholds is None:
        thresholds = np.arange(0.10, 0.91, 0.05)
    else:
        thresholds = np.asarray(thresholds, dtype=float).ravel()

    best_threshold = 0.5       # sensible default
    best_score = -1.0          # any real score will beat this

    for t in thresholds:
        # Binarize: scores >= threshold → 1, else → 0
        y_pred = (y_scores >= t).astype(int)
        score = metric_fn(y_true, y_pred)

        if score > best_score:
            best_score = score
            best_threshold = float(t)

    return best_threshold, best_score


# Classification Report

def classification_report(
    y_true: ArrayLike,
    y_pred: ArrayLike,
    class_names: Optional[List[str]] = None,
    print_report: bool = True,
) -> str:
    """Generate a formatted classification report similar to sklearn's.

    Produces per-class precision / recall / F1 / support as well as overall
    accuracy.

    """
    y_true, y_pred = _to_array(y_true), _to_array(y_pred)

    if class_names is None:
        class_names = ["Normal", "Anomaly"]

    tp, fp, fn, tn = _counts(y_true, y_pred)
    n = len(y_true)

    # ── Per-class metrics ─────────────────────────────────────────────────
    # Class 0 ("Normal"): treat class-0 as the "positive" class locally.
    #   precision_0 = TN / (TN + FN)   — of predicted-0, how many were 0?
    #   recall_0    = TN / (TN + FP)   — of actual-0, how many did we get?
    support_0 = tn + fp  # number of actual class-0 samples
    prec_0 = tn / (tn + fn) if (tn + fn) > 0 else 0.0
    rec_0 = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    f1_0 = 2 * prec_0 * rec_0 / (prec_0 + rec_0) if (prec_0 + rec_0) > 0 else 0.0

    # Class 1 ("Anomaly"): the standard TP / FP / FN definitions.
    support_1 = tp + fn  # number of actual class-1 samples
    prec_1 = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec_1 = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1_1 = 2 * prec_1 * rec_1 / (prec_1 + rec_1) if (prec_1 + rec_1) > 0 else 0.0

    # ── Overall metrics ───────────────────────────────────────────────────
    acc = (tp + tn) / n if n > 0 else 0.0
    macro_prec = (prec_0 + prec_1) / 2
    macro_rec = (rec_0 + rec_1) / 2
    macro_f1 = (f1_0 + f1_1) / 2
    weighted_prec = (prec_0 * support_0 + prec_1 * support_1) / n if n > 0 else 0.0
    weighted_rec = (rec_0 * support_0 + rec_1 * support_1) / n if n > 0 else 0.0
    weighted_f1 = (f1_0 * support_0 + f1_1 * support_1) / n if n > 0 else 0.0

    # ── Build the table ───────────────────────────────────────────────────
    header = f"{'':>15s} {'precision':>10s} {'recall':>10s} {'f1-score':>10s} {'support':>10s}"
    sep = "─" * len(header)

    rows = [
        sep,
        header,
        sep,
        f"{class_names[0]:>15s} {prec_0:10.4f} {rec_0:10.4f} {f1_0:10.4f} {support_0:10d}",
        f"{class_names[1]:>15s} {prec_1:10.4f} {rec_1:10.4f} {f1_1:10.4f} {support_1:10d}",
        sep,
        f"{'accuracy':>15s} {'':>10s} {'':>10s} {acc:10.4f} {n:10d}",
        f"{'macro avg':>15s} {macro_prec:10.4f} {macro_rec:10.4f} {macro_f1:10.4f} {n:10d}",
        f"{'weighted avg':>15s} {weighted_prec:10.4f} {weighted_rec:10.4f} {weighted_f1:10.4f} {n:10d}",
        sep,
    ]

    report = "\n".join(rows)

    if print_report:
        print(report)

    return report



In [ ]:
"""
utils/visualization.py —  for the LNN Telemetry Engine.


Every plot produced by this module should look *publication-ready* out of the
box: dark backgrounds, carefully chosen accent colours, generous whitespace,
and high-DPI output (300 dpi PNGs).

Colour palette
--------------
We use a curated set of accent colours that pop against a near-black canvas:

    TEAL      #00D9C0   — primary sensor traces / normal data
    ORANGE    #FF6F3C   — anomaly highlights / warnings
    PURPLE    #B37FEB   — secondary traces / model predictions
    MAGENTA   #FF4DA6   — false negatives / critical misses
    GOLD      #F5C542   — F1 / validation curves
    SLATE     #5E6B7A   — muted grid / annotations

Every function follows the same pattern:
1. Create figure + axes with the dark theme applied.
2. Draw the data.
3. Save to *save_path* (defaulting to ``outputs/figures/``).
4. Close the figure to free memory.

"""

from __future__ import annotations

import os
from pathlib import Path
from typing import List, Optional, Sequence, Union

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np

# Global configuration

# ── Colour palette ────────────────────────────────────────────────────────────
TEAL = "#00D9C0"
ORANGE = "#FF6F3C"
PURPLE = "#B37FEB"
MAGENTA = "#FF4DA6"
GOLD = "#F5C542"
SLATE = "#5E6B7A"
SOFT_RED = "#E8524A"
CYAN = "#56D6F5"

# A few extra colours for multi-line plots (hidden-state dimensions).
HIDDEN_STATE_PALETTE = [TEAL, ORANGE, PURPLE, GOLD, MAGENTA, CYAN]

# ── Default output directory ──────────────────────────────────────────────────
DEFAULT_FIGURE_DIR = os.path.join("outputs", "figures")

# ── DPI for saved PNGs ───────────────────────────────────────────────────────
SAVE_DPI = 300

# Type alias for convenience.
ArrayLike = Union[np.ndarray, List[float], Sequence[float]]


# Shared helpers

def _ensure_dir(path: str) -> None:
    """Create the parent directory for *path* if it doesn't already exist."""
    directory = os.path.dirname(path)
    if directory:
        os.makedirs(directory, exist_ok=True)


def _resolve_save_path(save_path: Optional[str], default_name: str) -> str:
    """Return a fully-resolved save path, falling back to the default dir."""
    if save_path is None:
        save_path = os.path.join(DEFAULT_FIGURE_DIR, default_name)
    _ensure_dir(save_path)
    return save_path


def _apply_dark_theme() -> None:
    """Apply a consistent dark theme to all matplotlib figures.

    We override individual rcParams instead of using plt.style.use() so
    that our theme is deterministic regardless of the user's matplotlibrc.
    """
    dark_params = {
        # Canvas & axes
        "figure.facecolor": "#0D1117",
        "axes.facecolor": "#161B22",
        "axes.edgecolor": SLATE,
        "axes.labelcolor": "#C9D1D9",
        "axes.titlesize": 14,
        "axes.titleweight": "bold",
        "axes.labelsize": 11,
        # Ticks
        "xtick.color": "#8B949E",
        "ytick.color": "#8B949E",
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        # Grid
        "axes.grid": True,
        "grid.color": "#21262D",
        "grid.linewidth": 0.6,
        "grid.alpha": 0.7,
        # Legend
        "legend.facecolor": "#161B22",
        "legend.edgecolor": SLATE,
        "legend.fontsize": 9,
        "legend.labelcolor": "#C9D1D9",
        # Text
        "text.color": "#C9D1D9",
        # Savefig
        "savefig.facecolor": "#0D1117",
        "savefig.edgecolor": "#0D1117",
        # Font
        "font.family": "sans-serif",
        "font.sans-serif": ["Inter", "Helvetica Neue", "Arial", "sans-serif"],
    }
    plt.rcParams.update(dark_params)


def _save_and_close(fig: plt.Figure, save_path: str) -> None:
    """Save a figure as a high-DPI PNG and close it to free memory."""
    fig.savefig(save_path, dpi=SAVE_DPI, bbox_inches="tight")
    plt.close(fig)
    print(f"   Figure saved → {save_path}")


def _highlight_anomalies(
    ax: plt.Axes,
    labels: np.ndarray,
    color: str = ORANGE,
    alpha: float = 0.15,
) -> None:
    """Shade contiguous anomaly regions (label == 1) on *ax*.

    We find the start/end indices of every contiguous run of 1s and draw
    a vertical span (axvspan) for each.
    """
    # Pad with 0s so np.diff catches edges at the boundaries.
    padded = np.concatenate([[0], labels, [0]])
    diff = np.diff(padded)

    # Rising edge → anomaly starts; falling edge → anomaly ends.
    starts = np.where(diff == 1)[0]
    ends = np.where(diff == -1)[0]

    for s, e in zip(starts, ends):
        ax.axvspan(s, e, color=color, alpha=alpha, zorder=0)


# 1. Sensor Anomaly Plot

def plot_sensor_anomalies(
    sensor_data: np.ndarray,
    anomaly_labels: ArrayLike,
    sensor_names: Optional[List[str]] = None,
    save_path: Optional[str] = None,
) -> str:
    """Multi-panel plot showing each sensor channel with anomaly highlights.

    """
    _apply_dark_theme()

    sensor_data = np.asarray(sensor_data)
    anomaly_labels = np.asarray(anomaly_labels, dtype=int).ravel()

    # Infer number of channels.
    if sensor_data.ndim == 1:
        sensor_data = sensor_data.reshape(-1, 1)
    n_channels = sensor_data.shape[1]

    if sensor_names is None:
        sensor_names = [f"Sensor {i}" for i in range(n_channels)]

    save_path = _resolve_save_path(save_path, "sensor_anomalies.png")

    # ── Create subplots — one row per channel ─────────────────────────────
    fig, axes = plt.subplots(
        n_channels, 1,
        figsize=(16, 2.8 * n_channels),
        sharex=True,
    )

    # Handle the case where there's only one sensor (axes is not a list).
    if n_channels == 1:
        axes = [axes]

    time_idx = np.arange(sensor_data.shape[0])

    for i, ax in enumerate(axes):
        # Plot the sensor trace in teal.
        ax.plot(time_idx, sensor_data[:, i], color=TEAL, linewidth=0.8, alpha=0.9)

        # Highlight anomaly regions in orange.
        _highlight_anomalies(ax, anomaly_labels, color=ORANGE, alpha=0.20)

        # Labels.
        ax.set_ylabel(sensor_names[i], fontsize=10, fontweight="bold")
        ax.tick_params(axis="both", which="major", labelsize=8)

    # Bottom axis gets the shared x-label.
    axes[-1].set_xlabel("Timestep", fontsize=11)
    fig.suptitle(
        "Sensor Channels with Anomaly Regions",
        fontsize=16, fontweight="bold", color="#E6EDF3", y=0.98,
    )
    fig.tight_layout(rect=[0, 0, 1, 0.96])

    _save_and_close(fig, save_path)
    return save_path


# 2. Hidden State Evolution (the "money shot")

def plot_hidden_states(
    hidden_states: np.ndarray,
    anomaly_labels: ArrayLike,
    save_path: Optional[str] = None,
    n_dims: int = 6,
) -> str:
    """Plot the evolution of liquid hidden-state dimensions over time.

     — it shows how the liquid neural network's
    internal state responds to anomalous conditions in the telemetry stream.

    """
    _apply_dark_theme()

    hidden_states = np.asarray(hidden_states)
    anomaly_labels = np.asarray(anomaly_labels, dtype=int).ravel()
    save_path = _resolve_save_path(save_path, "hidden_states.png")

    # Only plot the first `n_dims` dimensions (or fewer if D is small).
    n_dims = min(n_dims, hidden_states.shape[1])
    time_idx = np.arange(hidden_states.shape[0])

    fig, ax = plt.subplots(figsize=(16, 6))

    # ── Draw each hidden dimension with a unique colour ───────────────────
    for d in range(n_dims):
        colour = HIDDEN_STATE_PALETTE[d % len(HIDDEN_STATE_PALETTE)]
        ax.plot(
            time_idx,
            hidden_states[:, d],
            color=colour,
            linewidth=1.0,
            alpha=0.85,
            label=f"h[{d}]",
        )

    # ── Anomaly highlights ────────────────────────────────────────────────
    _highlight_anomalies(ax, anomaly_labels, color=SOFT_RED, alpha=0.18)

    # ── Aesthetics ────────────────────────────────────────────────────────
    ax.set_xlabel("Timestep", fontsize=12)
    ax.set_ylabel("Hidden State Value", fontsize=12)
    ax.set_title(
        "Liquid Neural Network — Hidden State Evolution",
        fontsize=16, fontweight="bold", color="#E6EDF3",
    )
    ax.legend(
        loc="upper right",
        ncol=n_dims,
        framealpha=0.6,
        fontsize=9,
    )

    fig.tight_layout()
    _save_and_close(fig, save_path)
    return save_path


# 3. Training Curves

def plot_training_curves(
    train_losses: ArrayLike,
    val_losses: ArrayLike,
    train_f1s: ArrayLike,
    val_f1s: ArrayLike,
    save_path: Optional[str] = None,
) -> str:
    """Side-by-side loss and F1 curves for training & validation.

    """
    _apply_dark_theme()
    save_path = _resolve_save_path(save_path, "training_curves.png")

    epochs = np.arange(1, len(train_losses) + 1)

    fig, (ax_loss, ax_f1) = plt.subplots(1, 2, figsize=(16, 5.5))

    # ── Left panel: Loss curves ───────────────────────────────────────────
    ax_loss.plot(epochs, train_losses, color=TEAL, linewidth=1.8, label="Train Loss")
    ax_loss.plot(
        epochs, val_losses,
        color=ORANGE, linewidth=1.8, linestyle="--", label="Val Loss",
    )
    # Small circle markers on each epoch for readability.
    ax_loss.scatter(epochs, train_losses, color=TEAL, s=18, zorder=5)
    ax_loss.scatter(epochs, val_losses, color=ORANGE, s=18, zorder=5)

    ax_loss.set_xlabel("Epoch", fontsize=12)
    ax_loss.set_ylabel("Loss", fontsize=12)
    ax_loss.set_title("Training & Validation Loss", fontsize=14, fontweight="bold")
    ax_loss.legend(loc="upper right", fontsize=10)

    # ── Right panel: F1 curves ────────────────────────────────────────────
    ax_f1.plot(epochs, train_f1s, color=PURPLE, linewidth=1.8, label="Train F1")
    ax_f1.plot(
        epochs, val_f1s,
        color=GOLD, linewidth=1.8, linestyle="--", label="Val F1",
    )
    ax_f1.scatter(epochs, train_f1s, color=PURPLE, s=18, zorder=5)
    ax_f1.scatter(epochs, val_f1s, color=GOLD, s=18, zorder=5)

    ax_f1.set_xlabel("Epoch", fontsize=12)
    ax_f1.set_ylabel("F1 Score", fontsize=12)
    ax_f1.set_title("Training & Validation F1", fontsize=14, fontweight="bold")
    ax_f1.legend(loc="lower right", fontsize=10)

    fig.suptitle(
        "LNN Training Progress",
        fontsize=16, fontweight="bold", color="#E6EDF3", y=1.01,
    )
    fig.tight_layout()
    _save_and_close(fig, save_path)
    return save_path


# 4. Confusion Matrix Heatmap

def plot_confusion_matrix(
    cm: np.ndarray,
    save_path: Optional[str] = None,
    class_names: Optional[List[str]] = None,
) -> str:
    """Annotated heatmap of a 2×2 confusion matrix.

    """
    _apply_dark_theme()
    save_path = _resolve_save_path(save_path, "confusion_matrix.png")

    if class_names is None:
        class_names = ["Normal", "Anomaly"]

    cm = np.asarray(cm)

    fig, ax = plt.subplots(figsize=(6, 5.5))

    # ── Custom colormap: dark teal → bright teal ──────────────────────────
    cmap = mcolors.LinearSegmentedColormap.from_list(
        "dark_teal", ["#0D1117", "#0A4D44", TEAL], N=256,
    )

    im = ax.imshow(cm, interpolation="nearest", cmap=cmap, aspect="equal")

    # ── Annotate each cell with its count ─────────────────────────────────
    threshold = cm.max() / 2.0  # text colour flips at halfway point
    for i in range(2):
        for j in range(2):
            text_colour = "#0D1117" if cm[i, j] > threshold else "#E6EDF3"
            ax.text(
                j, i, f"{cm[i, j]:,}",
                ha="center", va="center",
                fontsize=22, fontweight="bold",
                color=text_colour,
            )

    # ── Axis labels ───────────────────────────────────────────────────────
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(class_names, fontsize=12)
    ax.set_yticklabels(class_names, fontsize=12)
    ax.set_xlabel("Predicted Label", fontsize=13, labelpad=10)
    ax.set_ylabel("True Label", fontsize=13, labelpad=10)
    ax.set_title("Confusion Matrix", fontsize=15, fontweight="bold", pad=12)

    # Subtle colorbar on the right.
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.ax.tick_params(labelsize=9)

    fig.tight_layout()
    _save_and_close(fig, save_path)
    return save_path


# 5. Detection Timeline

def plot_detection_timeline(
    sensor_data: ArrayLike,
    true_labels: ArrayLike,
    pred_labels: ArrayLike,
    save_path: Optional[str] = None,
    sensor_name: str = "Sensor 0",
) -> str:
    """Overlay true vs predicted anomalies on a single sensor trace.

    Colour code
    -----------
    * **True Positive (TP)**: Correctly detected anomaly — shaded *teal*.
    * **False Positive (FP)**: False alarm — shaded *orange*.
    * **False Negative (FN)**: Missed anomaly — shaded *magenta*.

    """
    _apply_dark_theme()
    save_path = _resolve_save_path(save_path, "detection_timeline.png")

    sensor_data = np.asarray(sensor_data, dtype=float).ravel()
    true_labels = np.asarray(true_labels, dtype=int).ravel()
    pred_labels = np.asarray(pred_labels, dtype=int).ravel()
    time_idx = np.arange(len(sensor_data))

    fig, ax = plt.subplots(figsize=(18, 5))

    # ── Background sensor trace ───────────────────────────────────────────
    ax.plot(
        time_idx, sensor_data,
        color=SLATE, linewidth=0.7, alpha=0.6, label=sensor_name,
    )

    # ── Build per-timestep outcome labels ─────────────────────────────────
    # TP = true==1 AND pred==1  →  teal
    # FP = true==0 AND pred==1  →  orange
    # FN = true==1 AND pred==0  →  magenta
    tp_mask = (true_labels == 1) & (pred_labels == 1)
    fp_mask = (true_labels == 0) & (pred_labels == 1)
    fn_mask = (true_labels == 1) & (pred_labels == 0)

    # Highlight contiguous regions for each outcome category.
    _highlight_anomalies(ax, tp_mask.astype(int), color=TEAL, alpha=0.30)
    _highlight_anomalies(ax, fp_mask.astype(int), color=ORANGE, alpha=0.30)
    _highlight_anomalies(ax, fn_mask.astype(int), color=MAGENTA, alpha=0.30)

    # ── Legend patches (since axvspan doesn't auto-legend nicely) ─────────
    from matplotlib.patches import Patch

    legend_elements = [
        plt.Line2D([0], [0], color=SLATE, linewidth=1, label=sensor_name),
        Patch(facecolor=TEAL, alpha=0.4, label="True Positive"),
        Patch(facecolor=ORANGE, alpha=0.4, label="False Positive"),
        Patch(facecolor=MAGENTA, alpha=0.4, label="False Negative"),
    ]
    ax.legend(
        handles=legend_elements,
        loc="upper right",
        fontsize=10,
        framealpha=0.7,
    )

    # ── Axis decoration ──────────────────────────────────────────────────
    ax.set_xlabel("Timestep", fontsize=12)
    ax.set_ylabel("Value", fontsize=12)
    ax.set_title(
        "Anomaly Detection Timeline — True vs Predicted",
        fontsize=15, fontweight="bold", color="#E6EDF3",
    )

    fig.tight_layout()
    _save_and_close(fig, save_path)
    return save_path



## 2. Synthetic Data Generation
Generates 10,000 timesteps of CubeSat telemetry with injected faults.


In [ ]:
#!/usr/bin/env python3
import argparse
import os
import sys
import numpy as np
import pandas as pd

# ==============================================================================
# THEORY: Synthetic Data Generation
# ------------------------------------------------------------------------------
# Why generate our own data? Because real-world satellite telemetry with *perfectly 
# labeled anomalies* is incredibly rare and usually classified by the military or NASA.
# 
# To train our Liquid Neural Network, we need two things:
# 1. Clean Data (normal satellite operations)
# 2. Anomalous Data (where things go wrong, like a battery failure)
#
# Our simulation creates realistic noise and orbital cycles (like heating up in 
# the sun and cooling down in the Earth's shadow). Then, we randomly inject 
# 4 types of failures into the data and label those exact timestamps with a '1'.
# ==============================================================================

CHANNEL_SPECS = {
    "battery_voltage": (3.0, 4.2, 0.02),       
    "solar_current":   (0.0, 0.5, 0.01),        
    "cpu_temp":        (20.0, 45.0, 0.5),        
    "panel_temp":      (-40.0, 60.0, 1.0),       
    "gyro_x":          (-1.0, 1.0, 0.05),        
    "gyro_y":          (-1.0, 1.0, 0.05),        
    "gyro_z":          (-1.0, 1.0, 0.05),        
    "magnetometer":    (-50.0, 50.0, 0.5),       
}

CHANNEL_NAMES = list(CHANNEL_SPECS.keys())


def _generate_baseline(low: float, high: float, noise_std: float, n: int, rng: np.random.Generator) -> np.ndarray:
    mid = rng.uniform(low + 0.2 * (high - low), high - 0.2 * (high - low))
    period = rng.uniform(800, 1200)
    phase  = rng.uniform(0, 2 * np.pi)
    amplitude = 0.15 * (high - low)  
    t = np.arange(n, dtype=np.float64)
    signal = mid + amplitude * np.sin(2 * np.pi * t / period + phase)
    signal += rng.normal(0.0, noise_std, size=n)
    return signal


# ==============================================================================
# FUNCTION THEORY: Anomaly Injection
# ------------------------------------------------------------------------------
# Here we mathematically simulate four common space hardware failures:
# 1. Voltage Sag: The battery suddenly drops power (maybe an eclipse started).
# 2. Thermal Spike: The CPU or Solar Panel overheats rapidly.
# 3. Spin Drift: A reaction wheel breaks and the satellite starts slowly spinning.
# 4. Sensor Dropout: Cosmic radiation flips a bit, causing the sensor to output 0.0 or NaN.
# ==============================================================================
def _inject_voltage_sag(data: np.ndarray, indices: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    col = CHANNEL_NAMES.index("battery_voltage")
    labels = np.zeros(len(data), dtype=np.int32)
    for idx in indices:
        duration = rng.integers(5, 21)
        end = min(idx + duration, len(data))
        drop = rng.uniform(0.5, 1.5)
        data[idx:end, col] -= drop
        labels[idx:end] = 1
    return labels

def _inject_thermal_spike(data: np.ndarray, indices: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    target = rng.choice(["cpu_temp", "panel_temp"])
    col = CHANNEL_NAMES.index(target)
    labels = np.zeros(len(data), dtype=np.int32)
    for idx in indices:
        duration = rng.integers(10, 31)
        end = min(idx + duration, len(data))
        spike_mag = rng.uniform(15.0, 40.0)
        profile = np.concatenate([
            np.linspace(0, spike_mag, (end - idx) // 2 + 1),
            np.linspace(spike_mag, 0, (end - idx) - (end - idx) // 2)
        ])[: end - idx]
        data[idx:end, col] += profile
        labels[idx:end] = 1
    return labels

def _inject_spin_drift(data: np.ndarray, indices: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    target = rng.choice(["gyro_x", "gyro_y", "gyro_z"])
    col = CHANNEL_NAMES.index(target)
    labels = np.zeros(len(data), dtype=np.int32)
    for idx in indices:
        duration = rng.integers(50, 201)
        end = min(idx + duration, len(data))
        drift_rate = rng.uniform(0.5, 3.0) / duration
        direction = rng.choice([-1, 1])
        drift = direction * drift_rate * np.arange(end - idx)
        data[idx:end, col] += drift
        labels[idx:end] = 1
    return labels

def _inject_sensor_dropout(data: np.ndarray, indices: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    col = rng.integers(0, len(CHANNEL_NAMES))
    labels = np.zeros(len(data), dtype=np.int32)
    for idx in indices:
        duration = rng.integers(5, 16)
        end = min(idx + duration, len(data))
        fill = 0.0 if rng.random() < 0.5 else np.nan
        data[idx:end, col] = fill
        labels[idx:end] = 1
    return labels


ANOMALY_INJECTORS = [
    _inject_voltage_sag,
    _inject_thermal_spike,
    _inject_spin_drift,
    _inject_sensor_dropout,
]


def generate_telemetry(num_samples: int = 10000, anomaly_ratio: float = 0.05, seed: int = 42, output_dir: str = "data/raw") -> dict:
    rng = np.random.default_rng(seed)

    print("[1/4] Generating clean baselines for {} channels …".format(len(CHANNEL_NAMES)))
    baselines = {}
    for name in CHANNEL_NAMES:
        low, high, noise = CHANNEL_SPECS[name]
        baselines[name] = _generate_baseline(low, high, noise, num_samples, rng)
    data_clean = np.column_stack([baselines[ch] for ch in CHANNEL_NAMES])

    print("[2/4] Building normal telemetry DataFrame …")
    timestamps = pd.date_range(start="2026-01-01T00:00:00", periods=num_samples, freq="1s")
    df_normal = pd.DataFrame(data_clean, columns=CHANNEL_NAMES)
    df_normal.insert(0, "timestamp", timestamps)
    df_normal["anomaly"] = 0  

    print("[3/4] Injecting anomalies (target ratio ≈ {:.1%}) …".format(anomaly_ratio))
    data_anom = data_clean.copy()
    labels = np.zeros(num_samples, dtype=np.int32)

    avg_event_len = 30
    n_events = max(1, int(num_samples * anomaly_ratio / avg_event_len))

    event_starts = rng.choice(np.arange(100, num_samples - 200), size=n_events, replace=False)
    event_starts.sort()

    for start_idx in event_starts:
        injector = rng.choice(ANOMALY_INJECTORS)
        event_labels = injector(data_anom, np.array([start_idx]), rng)
        labels = np.maximum(labels, event_labels)  

    df_anomalous = pd.DataFrame(data_anom, columns=CHANNEL_NAMES)
    df_anomalous.insert(0, "timestamp", timestamps)
    df_anomalous["anomaly"] = labels

    print("[4/4] Saving CSVs …")
    os.makedirs(output_dir, exist_ok=True)
    normal_path = os.path.join(output_dir, "telemetry_normal.csv")
    anomalous_path = os.path.join(output_dir, "telemetry_anomalous.csv")

    df_normal.to_csv(normal_path, index=False)
    df_anomalous.to_csv(anomalous_path, index=False)

    actual_anomaly_ratio = labels.sum() / num_samples
    stats = {
        "num_samples": num_samples, "num_channels": len(CHANNEL_NAMES),
        "anomaly_events": n_events, "anomalous_timesteps": int(labels.sum()),
        "actual_anomaly_ratio": actual_anomaly_ratio,
        "normal_path": normal_path, "anomalous_path": anomalous_path,
    }

    print("\n" + "=" * 60)
    print("  Telemetry Generation Complete")
    print("=" * 60)
    print(f"  Total timesteps      : {num_samples:,}")
    print(f"  Sensor channels      : {len(CHANNEL_NAMES)}")
    print(f"  Anomaly events       : {n_events}")
    print(f"  Anomalous timesteps  : {int(labels.sum()):,}  ({actual_anomaly_ratio:.2%})")
    print(f"  Normal CSV           : {normal_path}")
    print(f"  Anomalous CSV        : {anomalous_path}")
    print("=" * 60)

    print("\n  Per-channel statistics (anomalous dataset):")
    print(f"  {'Channel':<20s} {'Mean':>10s} {'Std':>10s} {'Min':>10s} {'Max':>10s}")
    print("  " + "-" * 54)
    for i, ch in enumerate(CHANNEL_NAMES):
        col = data_anom[:, i]
        print(f"  {ch:<20s} {np.nanmean(col):>10.4f} {np.nanstd(col):>10.4f} "
              f"{np.nanmin(col):>10.4f} {np.nanmax(col):>10.4f}")
    print()

    return {"normal_path": normal_path, "anomalous_path": anomalous_path, "stats": stats}



# Generate the data immediately in Colab
print("Generating synthetic telemetry data...")
generate_telemetry(num_samples=10000, anomaly_ratio=0.08, seed=42, output_dir="data/raw")



## 3. Data Loading & Preprocessing
Sliding window PyTorch Dataset and min-max normalisation.


In [ ]:
#!/usr/bin/env python3
# ==============================================================================
# THEORY: Time-Series Data Loading & Windowing
# ------------------------------------------------------------------------------
# Neural networks (especially recurrent ones) need data in a specific format.
# When working with time-series data like telemetry, we can't just feed in one 
# data point at a time and expect the network to know if it's anomalous. It 
# needs *context*.
#
# We solve this using a "Sliding Window". We take a chunk of time (e.g., 50 
# seconds), look at all the sensor data in that chunk, and ask the model: 
# "Is there an anomaly anywhere in this window?" Then we slide the window 
# forward by 1 second and ask again.
#
# We also have to be very careful about Data Leakage. We split our data 
# chronologically (first 70% for training, next 15% for validation, last 15% 
# for testing). We MUST NOT shuffle before splitting, otherwise the model 
# will learn from the "future" to predict the "past"!
# ==============================================================================

import os
from typing import Dict, Optional, Tuple

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset

FEATURE_COLUMNS = [
    "battery_voltage", "solar_current", "cpu_temp", "panel_temp",
    "gyro_x", "gyro_y", "gyro_z", "magnetometer",
]
LABEL_COLUMN = "anomaly"
PROCESSED_DIR = "data/processed"


# ==============================================================================
# CLASS THEORY: PyTorch Dataset
# ------------------------------------------------------------------------------
# PyTorch requires us to wrap our data in a `Dataset` class so it knows how 
# to grab a single item (a window of features and its label) during training.
# ==============================================================================
class TelemetryDataset(Dataset):
    def __init__(self, features: np.ndarray, labels: np.ndarray):
        super().__init__()
        assert len(features) == len(labels)
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.float32).unsqueeze(-1)

    def __len__(self) -> int:
        return len(self.features)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        return self.features[idx], self.labels[idx]


# ==============================================================================
# FUNCTION THEORY: Normalisation
# ------------------------------------------------------------------------------
# Neural networks hate large numbers. If CPU temperature is 60 and Solar Current 
# is 0.2, the network will think Temp is way more important just because it's 
# bigger. We use Min-Max Normalisation to squash everything between 0 and 1.
# CRITICAL: We only calculate min and max from the TRAINING set!
# ==============================================================================
def _compute_norm_params(data: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    mins = np.nanmin(data, axis=0)
    maxs = np.nanmax(data, axis=0)
    return mins, maxs

def _apply_normalisation(data: np.ndarray, mins: np.ndarray, maxs: np.ndarray) -> np.ndarray:
    ranges = maxs - mins
    ranges[ranges == 0] = 1.0
    normalised = (data - mins) / ranges
    normalised = np.nan_to_num(normalised, nan=0.0)
    return normalised

def save_norm_params(mins: np.ndarray, maxs: np.ndarray, output_dir: str = PROCESSED_DIR) -> str:
    os.makedirs(output_dir, exist_ok=True)
    path = os.path.join(output_dir, "norm_params.npz")
    np.savez(path, mins=mins, maxs=maxs)
    print(f"  [data_loader] Normalisation params saved → {path}")
    return path

def load_norm_params(input_dir: str = PROCESSED_DIR) -> Tuple[np.ndarray, np.ndarray]:
    path = os.path.join(input_dir, "norm_params.npz")
    loaded = np.load(path)
    return loaded["mins"], loaded["maxs"]


# ==============================================================================
# FUNCTION THEORY: Window Extraction
# ------------------------------------------------------------------------------
# This takes the huge continuous array of data and slices it into overlapping 
# chunks. If any point in the chunk is anomalous (label == 1), the whole chunk 
# is labeled as anomalous.
# ==============================================================================
def _extract_windows(data: np.ndarray, labels: np.ndarray, seq_len: int, stride: int) -> Tuple[np.ndarray, np.ndarray]:
    n_samples = len(data)
    windows = []
    window_labels = []

    for start in range(0, n_samples - seq_len + 1, stride):
        end = start + seq_len
        windows.append(data[start:end])
        window_labels.append(1 if labels[start:end].max() > 0 else 0)

    return np.array(windows), np.array(window_labels)


def _save_processed(arrays: Dict[str, np.ndarray], output_dir: str = PROCESSED_DIR) -> None:
    os.makedirs(output_dir, exist_ok=True)
    for name, arr in arrays.items():
        path = os.path.join(output_dir, f"{name}.npy")
        np.save(path, arr)
    print(f"  [data_loader] Processed arrays saved → {output_dir}/")

def _load_processed(input_dir: str = PROCESSED_DIR) -> Optional[Dict[str, np.ndarray]]:
    expected = [
        "train_features", "train_labels", "val_features", "val_labels",
        "test_features", "test_labels",
    ]
    arrays = {}
    for name in expected:
        path = os.path.join(input_dir, f"{name}.npy")
        if not os.path.exists(path):
            return None
        arrays[name] = np.load(path)
    return arrays


# ==============================================================================
# FUNCTION THEORY: Pipeline Execution
# ------------------------------------------------------------------------------
# This orchestrates everything: it loads the raw CSV, splits it chronologically, 
# normalises it, extracts windows, caches it to disk to save time on future runs, 
# and finally returns PyTorch DataLoaders ready for the training script!
# ==============================================================================
def get_dataloaders(
    csv_path: str = "data/raw/telemetry_anomalous.csv",
    seq_len: int = 50,
    batch_size: int = 64,
    stride: int = 1,
    train_ratio: float = 0.70,
    val_ratio: float = 0.15,
    processed_dir: str = PROCESSED_DIR,
    force_reprocess: bool = False,
    num_workers: int = 0,
) -> Tuple[DataLoader, DataLoader, DataLoader, dict]:
    
    cached = None if force_reprocess else _load_processed(processed_dir)

    if cached is not None:
        print("  [data_loader] Loading cached processed arrays …")
        train_features = cached["train_features"]
        train_labels   = cached["train_labels"]
        val_features   = cached["val_features"]
        val_labels     = cached["val_labels"]
        test_features  = cached["test_features"]
        test_labels    = cached["test_labels"]
        mins, maxs     = load_norm_params(processed_dir)
    else:
        print(f"  [data_loader] Reading CSV: {csv_path}")
        df = pd.read_csv(csv_path)

        features = df[FEATURE_COLUMNS].values.astype(np.float64)
        labels   = df[LABEL_COLUMN].values.astype(np.int32)

        n_total = len(features)
        print(f"  [data_loader] Total timesteps: {n_total:,}")

        train_end = int(n_total * train_ratio)
        val_end   = int(n_total * (train_ratio + val_ratio))

        feat_train = features[:train_end]
        feat_val   = features[train_end:val_end]
        feat_test  = features[val_end:]

        lab_train = labels[:train_end]
        lab_val   = labels[train_end:val_end]
        lab_test  = labels[val_end:]

        print(f"  [data_loader] Split sizes — "
              f"train: {len(feat_train):,}  "
              f"val: {len(feat_val):,}  "
              f"test: {len(feat_test):,}")

        mins, maxs = _compute_norm_params(feat_train)
        save_norm_params(mins, maxs, processed_dir)

        feat_train = _apply_normalisation(feat_train, mins, maxs)
        feat_val   = _apply_normalisation(feat_val, mins, maxs)
        feat_test  = _apply_normalisation(feat_test, mins, maxs)

        print(f"  [data_loader] Extracting windows (seq_len={seq_len}, stride={stride}) …")

        train_features, train_labels = _extract_windows(feat_train, lab_train, seq_len, stride)
        val_features, val_labels = _extract_windows(feat_val, lab_val, seq_len, stride)
        test_features, test_labels = _extract_windows(feat_test, lab_test, seq_len, stride)

        print(f"  [data_loader] Window counts — "
              f"train: {len(train_features):,}  "
              f"val: {len(val_features):,}  "
              f"test: {len(test_features):,}")

        _save_processed({
            "train_features": train_features,
            "train_labels":   train_labels,
            "val_features":   val_features,
            "val_labels":     val_labels,
            "test_features":  test_features,
            "test_labels":    test_labels,
        }, processed_dir)

    train_ds = TelemetryDataset(train_features, train_labels)
    val_ds   = TelemetryDataset(val_features, val_labels)
    test_ds  = TelemetryDataset(test_features, test_labels)

    train_loader = DataLoader(
        train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers, drop_last=False
    )
    val_loader = DataLoader(
        val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, drop_last=False
    )
    test_loader = DataLoader(
        test_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, drop_last=False
    )

    metadata = {
        "seq_len":    seq_len,
        "n_features": train_features.shape[-1],
        "train_size": len(train_ds),
        "val_size":   len(val_ds),
        "test_size":  len(test_ds),
        "mins":       mins,
        "maxs":       maxs,
    }

    print(f"\n  [data_loader] DataLoaders ready  (batch_size={batch_size}, seq_len={seq_len})")
    print(f"  [data_loader]   Train batches : {len(train_loader)}")
    print(f"  [data_loader]   Val batches   : {len(val_loader)}")
    print(f"  [data_loader]   Test batches  : {len(test_loader)}")

    return train_loader, val_loader, test_loader, metadata





## 4. Model Architecture (LNN)
Fully-connected Liquid Neural Network (CfC).


In [ ]:
# ==============================================================================
# THEORY: Brain-Inspired Sparse Wiring (AutoNCP)
# ------------------------------------------------------------------------------
# In traditional deep learning, layers are "dense"—meaning every neuron connects 
# to every neuron in the next layer. This uses huge amounts of memory.
#
# Biologists mapping the brain of the C. elegans worm discovered that its 302 
# neurons are wired sparsely, organized into specific layers:
#   Sensory (Input) -> Inter (Hidden) -> Command (Decision) -> Motor (Output)
#
# Neural Circuit Policies (NCP) is a framework that mimics this biological 
# wiring. Instead of dense matrix multiplication, we use a sparse topology.
# This results in incredibly tiny models (great for microcontrollers!) that 
# don't overfit easily, because the structure naturally acts as a regularizer.
# ==============================================================================

import os
import sys

from ncps.wirings import AutoNCP 


# ==============================================================================
# FUNCTION THEORY: Building the Wiring
# ------------------------------------------------------------------------------
# The AutoNCP class automatically figures out the best way to distribute our 
# neurons across the Sensory, Inter, Command, and Motor categories, and connects 
# them sparsely.
# ==============================================================================
def build_wiring(units: int = 32, output_size: int = 1) -> AutoNCP:
    wiring = AutoNCP(units=units, output_size=output_size)
    return wiring


# ==============================================================================
# FUNCTION THEORY: Inspecting the Topology
# ------------------------------------------------------------------------------
# Before we train, it's cool to look at exactly how many synapses (connections) 
# were generated compared to a traditional dense network.
# ==============================================================================
def get_wiring_summary(wiring: AutoNCP) -> dict:
    total_neurons = wiring.units
    motor_neurons = wiring.output_dim

    try:
        import numpy as np
        adj = np.array(wiring.adjacency_matrix)
        total_synapses = int(np.count_nonzero(adj))
    except (AttributeError, Exception):
        total_synapses = -1  

    summary = {
        "total_neurons": total_neurons,
        "motor_neurons": motor_neurons,
        "total_synapses": total_synapses,
    }

    print("=" * 55)
    print("  LNN Telemetry Engine — Wiring Summary")
    print("=" * 55)
    print(f"  Total neurons (units)  : {total_neurons}")
    print(f"  Motor neurons (output) : {motor_neurons}")
    if total_synapses >= 0:
        print(f"  Total synapses         : {total_synapses}")
        max_synapses = total_neurons * total_neurons
        sparsity = 1.0 - (total_synapses / max_synapses) if max_synapses > 0 else 0.0
        print(f"  Sparsity               : {sparsity:.1%}")
    else:
        print("  Total synapses         : (unavailable before binding)")
    print("=" * 55)

    return summary


# ==============================================================================
# FUNCTION THEORY: Visualization
# ------------------------------------------------------------------------------
# This function draws the brain-like structure of our network so we can see 
# the sparse connections visually!
# ==============================================================================
def visualize_wiring(wiring: AutoNCP, save_path: str = "wiring_diagram.png") -> None:
    import matplotlib
    matplotlib.use("Agg")  
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches

    if hasattr(wiring, "draw_graph"):
        try:
            plt.figure(figsize=(10, 6))
            wiring.draw_graph()
            plt.title("AutoNCP Sparse Wiring — C. elegans Inspired", fontsize=14)
            plt.tight_layout()
            plt.savefig(save_path, dpi=150, bbox_inches="tight")
            plt.close()
            print(f"[] Wiring diagram saved to: {save_path}")
            return
        except Exception as e:
            print(f"[!] draw_graph() failed ({e}), falling back to manual diagram.")

    fig, ax = plt.subplots(figsize=(12, 7))
    ax.set_xlim(-0.5, 4.5)
    ax.set_ylim(-1, 8)
    ax.set_aspect("equal")
    ax.axis("off")
    fig.patch.set_facecolor("#0d1117")
    ax.set_facecolor("#0d1117")

    n = wiring.units
    motor = wiring.output_dim
    inter = max(1, n // 3)
    command = max(1, n // 3)
    sensory = n - inter - command - motor

    layers = [
        (0.5, "Sensory\n(Input)", max(sensory, 1), "#58a6ff"),
        (1.7, "Inter\n(Hidden)", max(inter, 1), "#3fb950"),
        (2.9, "Command\n(Decision)", max(command, 1), "#d29922"),
        (4.1, "Motor\n(Output)", motor, "#f85149"),
    ]

    for x, label, count, color in layers:
        y_start = (7 - count) / 2
        for i in range(min(count, 7)):  
            y = y_start + i * 0.9
            circle = plt.Circle((x, y), 0.3, color=color, alpha=0.75, zorder=3)
            ax.add_patch(circle)
        if count > 7:
            ax.text(x, y_start - 0.7, f"(+{count - 7} more)", ha="center",
                    fontsize=8, color="white", alpha=0.7)
        ax.text(x, -0.5, label, ha="center", fontsize=10, color="white",
                fontweight="bold")
        ax.text(x, -1.0, f"n={count}", ha="center", fontsize=9, color="gray")

    for i in range(len(layers) - 1):
        x1 = layers[i][0] + 0.35
        x2 = layers[i + 1][0] - 0.35
        y_mid = 3.5
        ax.annotate("", xy=(x2, y_mid), xytext=(x1, y_mid),
                     arrowprops=dict(arrowstyle="->", color="white",
                                     lw=2, alpha=0.5))

    ax.set_title("AutoNCP Sparse Wiring — C. elegans Inspired",
                 fontsize=14, color="white", pad=20)
    legend_handles = [
        mpatches.Patch(color=c, label=l.split("\n")[0])
        for _, l, _, c in layers
    ]
    ax.legend(handles=legend_handles, loc="upper right", fontsize=9,
              facecolor="#161b22", edgecolor="gray", labelcolor="white")

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.close()
    print(f"[] Wiring diagram saved to: {save_path}")


if __name__ == "__main__":
    print("\n  Building C. elegans-inspired sparse wiring...\n")

    wiring = build_wiring(units=32, output_size=1)
    summary = get_wiring_summary(wiring)

    output_path = os.path.join(os.path.dirname(__file__), "wiring_diagram.png")
    print(f"\n  Generating wiring visualization...")
    visualize_wiring(wiring, save_path=output_path)

    print(f"\n✅  Done!  Summary dict: {summary}")


# ==============================================================================
# THEORY: Liquid Neural Networks (LNNs)
# ------------------------------------------------------------------------------
# Welcome to the core of the engine! 
# 
# Standard neural networks (like CNNs or LSTMs) operate on discrete time steps.
# Liquid Neural Networks (LNNs) are different. They are continuous-time recurrent 
# neural networks inspired by the tiny brain of the C. elegans worm.
#
# The hidden state of an LNN isn't just updated by a simple math function; 
# it's governed by an Ordinary Differential Equation (ODE). This allows the 
# network to process incoming data at any irregular time interval and adapt 
# dynamically to the environment (hence the name "Liquid").
#
# Specifically, we use the Closed-form Continuous-depth (CfC) architecture. 
# Solving ODEs numerically (like with standard Neural ODEs) is very slow. 
# CfCs use a mathematical closed-form solution to approximate the ODE, giving 
# us the power of a continuous-time network but running as fast as an LSTM!
# ==============================================================================

import torch
import torch.nn as nn

from ncps.torch import CfC
from ncps.wirings import AutoNCP




class LNNAnomalyDetector(nn.Module):
    # ==========================================================================
    # CLASS THEORY: The Anomaly Detector
    # --------------------------------------------------------------------------
    # This class wraps the CfC network. It takes in a sequence of telemetry 
    # data (like battery voltage and temperature over time) and outputs a single 
    # score between 0 and 1. 
    # 0 means "Normal", 1 means "Anomaly".
    # ==========================================================================
    
    def __init__(
        self,
        input_size: int,
        units: int = 32,
        output_size: int = 1,
        use_sparse_wiring: bool = True,
    ):
        super().__init__()

        self.input_size = input_size
        self.units = units
        self.output_size = output_size
        self.use_sparse_wiring = use_sparse_wiring

        # ----------------------------------------------------------------------
        # THEORY: Wiring Topology
        # We can either wire every neuron to every other neuron (Dense), or we
        # can wire them sparsely like a real brain (Sparse/AutoNCP). Sparse 
        # wiring saves a huge amount of memory and prevents overfitting.
        # ----------------------------------------------------------------------
        if use_sparse_wiring:
            self.wiring = build_wiring(units=units, output_size=output_size)
            self.backbone = CfC(input_size=input_size, units=self.wiring)
            self.output_head = nn.Sigmoid()
            self._backbone_out_dim = output_size
        else:
            self.wiring = None
            self.backbone = CfC(input_size=input_size, units=units)
            self.output_head = nn.Sequential(
                nn.Linear(units, output_size),
                nn.Sigmoid()
            )
            self._backbone_out_dim = units

    
    # ==========================================================================
    # FUNCTION THEORY: The Forward Pass
    # --------------------------------------------------------------------------
    # When data passes through the network, the CfC backbone calculates the
    # hidden state for every single time step. 
    # However, since we are doing sequence classification (looking at a whole 
    # window of data to make one decision), we only care about the very LAST 
    # time step's output to make our anomaly prediction!
    # ==========================================================================
    def forward(
        self,
        x: torch.Tensor,
        hidden: torch.Tensor = None,
        return_hidden: bool = False,
    ):
        if hidden is not None:
            backbone_out, hn = self.backbone(x, hx=hidden)
        else:
            backbone_out, hn = self.backbone(x)

        # Grab the last timestep 
        last_timestep = backbone_out[:, -1, :] 

        # Apply the output head to get our prediction
        predictions = self.output_head(last_timestep)

        if return_hidden:
            return predictions, backbone_out
        else:
            return predictions

    
    # ==========================================================================
    # FUNCTION THEORY: Parameter Counting
    # --------------------------------------------------------------------------
    # For edge computing on satellites or rovers, we have strict memory limits. 
    # This helper counts exactly how many trainable weights the model has, so 
    # we can calculate its RAM footprint.
    # ==========================================================================
    def count_parameters(self) -> int:
        total = sum(p.numel() for p in self.parameters() if p.requires_grad)
        return total

    def model_summary(self) -> None:
        print("\n" + "=" * 60)
        print("  LNN Telemetry Engine — Model Summary")
        print("=" * 60)

        mode = "Sparse (AutoNCP)" if self.use_sparse_wiring else "Dense (Fully Connected)"
        print(f"\n  Mode             : {mode}")
        print(f"  Input features   : {self.input_size}")
        print(f"  CfC neurons      : {self.units}")
        print(f"  Output size      : {self.output_size}")
        print(f"  Backbone out dim : {self._backbone_out_dim}")

        total_params = self.count_parameters()
        memory_fp32_kb = (total_params * 4) / 1024  
        memory_int8_kb = (total_params * 1) / 1024  

        print(f"\n  Trainable params : {total_params:,}")
        print(f"  Memory (FP32)    : {memory_fp32_kb:.1f} KB")
        print(f"  Memory (INT8)    : {memory_int8_kb:.1f} KB")

        print(f"\n  Layer Breakdown:")
        print(f"  {'-' * 50}")
        for name, module in self.named_children():
            n_params = sum(p.numel() for p in module.parameters() if p.requires_grad)
            print(f"    {name:20s} → {n_params:>8,} params  ({type(module).__name__})")

        lstm_params = 4 * (self.input_size * self.units + self.units * self.units + self.units)
        ratio = lstm_params / total_params if total_params > 0 else float("inf")
        print(f"\n  Comparable LSTM  : ~{lstm_params:,} params ({ratio:.1f}× larger)")
        print(f"    LNN is {ratio:.1f}× more parameter-efficient than LSTM")

        print("\n" + "=" * 60)


if __name__ == "__main__":
    print("\n  LNN Core — Smoke Test\n")

    print("=" * 60)
    print("  TEST 1: Sparse Wiring (AutoNCP)")
    print("=" * 60)

    model_sparse = LNNAnomalyDetector(input_size=8, units=32, use_sparse_wiring=True)
    model_sparse.model_summary()

    dummy_input = torch.randn(4, 50, 8)
    preds = model_sparse(dummy_input)
    print(f"\n  Input shape      : {dummy_input.shape}")
    print(f"  Output shape     : {preds.shape}")
    print(f"  Output range     : [{preds.min().item():.4f}, {preds.max().item():.4f}]")

    preds_h, hidden_states = model_sparse(dummy_input, return_hidden=True)
    print(f"  Hidden states    : {hidden_states.shape}")

    print("\n" + "=" * 60)
    print("  TEST 2: Fully Connected (Dense Baseline)")
    print("=" * 60)

    model_dense = LNNAnomalyDetector(input_size=8, units=32, use_sparse_wiring=False)
    model_dense.model_summary()

    preds_dense = model_dense(dummy_input)
    print(f"\n  Input shape      : {dummy_input.shape}")
    print(f"  Output shape     : {preds_dense.shape}")
    print(f"  Output range     : [{preds_dense.min().item():.4f}, {preds_dense.max().item():.4f}]")

    print("\n✅  All smoke tests passed!")



## 5. Training Loop
Train the model using the weighted BCE loss and cosine annealing.


In [ ]:

# ==========================================
# CONFIGURATION
# ==========================================
csv_path = "data/raw/telemetry_anomalous.csv"
processed_dir = "data/processed"
seq_len = 50
stride = 5
batch_size = 64
train_ratio = 0.70
val_ratio = 0.15

input_size = 8
units = 48
output_size = 1

epochs = 60
learning_rate = 0.003
weight_decay = 0.0001
patience = 15

# ==========================================
# 1. LOAD DATA
# ==========================================
train_loader, val_loader, test_loader, data_meta = get_dataloaders(
    csv_path=csv_path,
    processed_dir=processed_dir,
    seq_len=seq_len,
    stride=stride,
    batch_size=batch_size,
    train_ratio=train_ratio,
    val_ratio=val_ratio,
    force_reprocess=True
)

# ==========================================
# 2. INITIALIZE MODEL
# ==========================================
model = LNNAnomalyDetector(
    input_size=input_size,
    units=units,
    output_size=output_size,
    use_sparse_wiring=False
).to(device)

model.model_summary()

# ==========================================
# 3. TRAINING SETUP
# ==========================================
train_labels = train_loader.dataset.labels
n_pos = train_labels.sum().item()
n_neg = len(train_labels) - n_pos
pos_weight_val = n_neg / max(n_pos, 1)

def weighted_bce(output, target):
    weights = torch.where(target == 1, pos_weight_val, 1.0)
    bce = nn.functional.binary_cross_entropy(output, target, reduction='none')
    return (bce * weights).mean()

criterion = weighted_bce
optimizer = AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
scheduler = CosineAnnealingLR(optimizer, T_max=epochs)

# ==========================================
# 4. TRAINING LOOP
# ==========================================
best_val_f1 = 0.0
patience_counter = 0
history = {"train_loss": [], "val_loss": [], "train_f1": [], "val_f1": []}

print("\nStarting training...")
for epoch in range(1, epochs + 1):
    # Train
    model.train()
    running_loss = 0.0
    all_preds, all_labels = [], []
    
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        preds = model(x)
        loss = criterion(preds, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        running_loss += loss.item() * x.size(0)
        all_preds.extend((preds.detach().cpu().numpy() >= 0.5).astype(int))
        all_labels.extend(y.cpu().numpy())
        
    train_loss = running_loss / len(train_loader.dataset)
    train_f1 = f1_score(all_labels, all_preds)
    
    # Validate
    model.eval()
    val_loss = 0.0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            preds = model(x)
            loss = criterion(preds, y)
            val_loss += loss.item() * x.size(0)
            all_preds.extend((preds.cpu().numpy() >= 0.5).astype(int))
            all_labels.extend(y.cpu().numpy())
            
    val_loss = val_loss / len(val_loader.dataset)
    val_f1 = f1_score(all_labels, all_preds)
    
    scheduler.step()
    
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_f1"].append(train_f1)
    history["val_f1"].append(val_f1)
    
    if val_f1 >= best_val_f1:
        best_val_f1 = val_f1
        patience_counter = 0
        best_model_state = model.state_dict()
        marker = "★ saved"
    else:
        patience_counter += 1
        marker = ""
        
    print(f"Epoch {epoch:2d}/{epochs} | Loss: {train_loss:.4f}/{val_loss:.4f} | F1: {train_f1:.4f}/{val_f1:.4f} {marker}")
    
    if patience_counter >= patience:
        print(f"\nEarly stopping at epoch {epoch}")
        break

# Restore best model
model.load_state_dict(best_model_state)

# Plot training curves
plot_training_curves(history["train_loss"], history["val_loss"], history["train_f1"], history["val_f1"], save_path="training_curves.png")
plt.show()



## 6. Evaluation
Run inference on the test set and plot results.


In [ ]:

# ==========================================
# 5. EVALUATION
# ==========================================
model.eval()
test_loss = 0.0
all_preds, all_labels, all_scores = [], [], []
hidden_states_list = []

print("Evaluating on test set...")
with torch.no_grad():
    for x, y in test_loader:
        x, y = x.to(device), y.to(device)
        preds, hidden = model(x, return_hidden=True)
        loss = criterion(preds, y)
        
        test_loss += loss.item() * x.size(0)
        scores = preds.cpu().numpy()
        all_scores.extend(scores)
        all_preds.extend((scores >= 0.5).astype(int))
        all_labels.extend(y.cpu().numpy())
        
        # Save hidden states for the first batch
        if len(hidden_states_list) == 0:
            hidden_states_list.append(hidden.cpu().numpy())

test_loss = test_loss / len(test_loader.dataset)

# Optimize threshold
best_thresh, best_f1 = optimize_threshold(all_labels, all_scores, metric="f1")
final_preds = (np.array(all_scores) >= best_thresh).astype(int)

print(f"\nOptimized threshold: {best_thresh:.2f} (F1: {best_f1:.4f})")
print("\nResults @ optimal threshold:")
classification_report(all_labels, final_preds)

# ==========================================
# 6. VISUALIZATIONS
# ==========================================
cm = confusion_matrix(all_labels, final_preds)
plot_confusion_matrix(cm, save_path="confusion_matrix.png")
plt.show()

# Extract first sample from test set for timeline
sample_x, sample_y = next(iter(test_loader))
sample_preds, _ = model(sample_x.to(device), return_hidden=True)
sample_preds_bin = (sample_preds.cpu().numpy() >= best_thresh).astype(int)

# Battery voltage is channel 0
plot_detection_timeline(
    sensor_data=sample_x.numpy()[:, -1, 0], 
    true_labels=sample_y.numpy(), 
    pred_labels=sample_preds_bin, 
    save_path="detection_timeline.png",
    sensor_name="Battery Voltage"
)
plt.show()

plot_hidden_states(
    hidden_states=hidden_states_list[0][0], # First sample in first batch
    anomaly_labels=sample_y.numpy()[0], 
    save_path="hidden_state_evolution.png"
)
plt.show()

